In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import polars as pl
import math
from typing import Dict, List, Optional

sys.path.append('../')

# corrected imports based on new project structure
from data.data_loading import load_all_raw_data
from plots.data_plot import (
    plot_user_analysis, plot_temporal_analysis, 
    plot_station_analysis, plot_activity_heatmap, 
    print_summary_statistics
)
from analysis.data_analysis import (
    analyze_users_for_visualization, 
    analyze_trips_for_visualization
)
from analysis.station_analytics import analyze_station_activity, compare_stations

from data.weather_processing import WeatherDataCollector
from features.feature_engineering import (
    apply_all_enrichments
)


In [2]:
trips_with_weather = pl.read_parquet('../../data/processed/trips_with_weather.parquet')

trips_with_weather = trips_with_weather.with_columns(
    pl.col('fecha_origen_recorrido').str.strptime(pl.Datetime, strict=False).alias('fecha_origen_recorrido')
)

trips_with_weather = trips_with_weather.filter(
    pl.col('fecha_origen_recorrido') >= pl.datetime(2021, 1, 1)
)

print(f"loaded {trips_with_weather.shape[0]:,} trips with weather data")
print(f"columns: {len(trips_with_weather.columns)}")
print(f"date range: {trips_with_weather['fecha_origen_recorrido'].min()} to {trips_with_weather['fecha_origen_recorrido'].max()}")



loaded 10,560,456 trips with weather data
columns: 48
date range: 2021-01-01 00:04:26 to 2024-08-30 23:57:59


In [ ]:
import os
import shutil
if os.path.exists('feature_checkpoints'):
    shutil.rmtree('feature_checkpoints')
    print("Cleared existing checkpoints")

import importlib
import sys

In [3]:

if 'features.core_pipeline' in sys.modules:
    importlib.reload(sys.modules['features.core_pipeline'])
from features.core_pipeline import engineer_ecobici_features

df_feat_test = engineer_ecobici_features(
    trips_with_weather,
    dt_minutes=30,                          
    max_memory_mb=15000,  
    chunk_size_days=10,   
    clear_checkpoints=True,               
    enable_streaming=False  
)

print(f"Output shape: {df_feat_test.shape}")
print(f"Columns: {len(df_feat_test.columns)}")
print(f"Memory usage: ~{df_feat_test.estimated_size('mb'):.1f} MB")

🚴‍♂️  MEMORY-OPTIMIZED Feature Engineering Pipeline
  Max memory limit: 15000 MB
  Chunk size: 10 days
  Streaming enabled: False
  -> Loading temporal features from checkpoint: C:\Users\xxx\Documents\GitHub\EcoBici-AI\RUNS\data\feature_checkpoints\step09_temporal_optimized.parquet
  -> Temporal checkpoint loaded, proceeding to final steps...
[Step 10/11] Final data cleaning and optimization...
  -> Optimizing data types for memory efficiency...
  -> Optimized 36 columns
  Memory usage after after data type optimization: 635.9 MB
[Step 11/11] Saving final result...
  -> Final feature dataset shape: (25613290, 73)
  -> Final dataset size: 7016.0 MB
  -> Large dataset detected, using chunked writing...
     -> Writing 25,613,290 rows in 513 chunks of 50,000 rows each
     -> Wrote chunk 2/513
     -> Wrote chunk 3/513
     -> Wrote chunk 4/513
     -> Wrote chunk 5/513
     -> Wrote chunk 6/513
     -> Wrote chunk 7/513
     -> Wrote chunk 8/513
     -> Wrote chunk 9/513
     -> Wrote ch

In [ ]:
#   # Save the result
# test_path = '../../data/processed/trips_feat_eng_iterative_test.parquet'
# df_feat_test.write_parquet(test_path)
# print(f"   💾 Saved to: {test_path}")

In [4]:
df_feat = pl.read_parquet(r"../../data/processed/trips_feat_eng.parquet")

In [5]:
df_feat.columns #agregar y_departures_next_DT

['station_id',
 'ts_start',
 'dep_last_DT',
 'trip_dur_mean_last_DT',
 'model_ICONIC_cnt',
 'share_male',
 'share_female',
 'share_other',
 'dep_lag_1',
 'dep_lag_2',
 'dep_lag_3',
 'dep_lag_4',
 'dep_lag_5',
 'dep_lag_6',
 'arr_last_DT',
 'arr_lag_1',
 'arr_lag_2',
 'arr_lag_3',
 'arr_lag_4',
 'arr_lag_5',
 'arr_lag_6',
 'y_arrivals_next_DT',
 'weather_temperature_2m',
 'weather_relative_humidity_2m',
 'weather_dew_point_2m',
 'weather_apparent_temperature',
 'weather_precipitation',
 'weather_rain',
 'weather_weather_code',
 'weather_pressure_msl',
 'weather_surface_pressure',
 'weather_cloud_cover',
 'weather_cloud_cover_low',
 'weather_cloud_cover_mid',
 'weather_cloud_cover_high',
 'weather_et0_fao_evapotranspiration',
 'weather_vapour_pressure_deficit',
 'weather_wind_speed_10m',
 'weather_wind_speed_100m',
 'weather_wind_direction_10m',
 'weather_wind_direction_100m',
 'weather_wind_gusts_10m',
 'weather_soil_temperature_0_to_7cm',
 'weather_soil_temperature_7_to_28cm',
 'weathe

In [6]:
# Obtener solo las columnas que tienen nulls (> 0)
df_nulls = df_feat.null_count()

null_summary = df_nulls.transpose(include_header=True)
null_summary.columns = ["column", "null_count"]

cols_with_nulls = null_summary.filter(pl.col("null_count") > 0)
print(cols_with_nulls)


shape: (47, 2)
┌─────────────────────────────────┬────────────┐
│ column                          ┆ null_count │
│ ---                             ┆ ---        │
│ str                             ┆ u32        │
╞═════════════════════════════════╪════════════╡
│ model_ICONIC_cnt                ┆ 24419376   │
│ share_male                      ┆ 24419376   │
│ share_female                    ┆ 24419376   │
│ share_other                     ┆ 24419376   │
│ dep_lag_1                       ┆ 24637416   │
│ …                               ┆ …          │
│ weather_soil_moisture_28_to_10… ┆ 219144     │
│ weather_soil_moisture_100_to_2… ┆ 219144     │
│ weather_sunshine_duration       ┆ 219144     │
│ weather_is_day                  ┆ 219144     │
│ weather_direct_radiation        ┆ 219144     │
└─────────────────────────────────┴────────────┘


In [9]:
# # analizar una estación específica
# results = analyze_station_activity(df_feat, station_id=2, show_plots=True)

# # comparar múltiples estaciones
# compare_stations(df_feat, station_ids=[2, 27, 3, 111], metric='dep_last_DT')

In [7]:

print("Loading raw data for coordinate information...")
trips_with_weather = pl.read_parquet('../../data/processed/trips_with_weather.parquet')

# Apply all enrichments
df_feat_enriched, station_id_to_idx, embedding_dim = apply_all_enrichments(
    df=df_feat,
    df_raw=trips_with_weather,
    add_coords=True,
    add_fourier=True,
    k_values=[1, 2]
)


new_cols = [col for col in df_feat_enriched.columns if col not in df_feat.columns]
print(f"   New columns: {new_cols}")


import pickle
with open('../../data/processed/station_id_to_idx.pkl', 'wb') as f:
    pickle.dump(station_id_to_idx, f)
    
with open('../../data/processed/embedding_info.pkl', 'wb') as f:
    pickle.dump({'embedding_dim': embedding_dim, 'n_stations': len(station_id_to_idx)}, f)

print(f"\nSaved embedding mapping ({len(station_id_to_idx)} stations, dim={embedding_dim})")

df_feat = df_feat_enriched.clone()


print("\n🔍 Checking for nulls in new columns:")
null_counts = df_feat.select([pl.col(col).null_count().alias(col) for col in new_cols]).to_dicts()[0]
for col, count in null_counts.items():
    if count > 0:
        print(f"   {col}: {count:,} nulls")
    else:
        print(f"   {col}: ✅ no nulls")


Loading raw data for coordinate information...
🚀 APPLYING ALL FEATURE ENRICHMENTS
Adding coordinates to feature dataframe...
Creating enhanced station metadata...
  -> Extracted metadata for 501 stations
  -> Successfully added coordinates to all 31,775,880 rows
Normalizing coordinates...
  -> Coordinate normalization stats:
     lat: mean=-34.601450, std=0.027630
     lon: mean=-58.428608, std=0.039224
Adding fourier coordinates with k_values=[1, 2]...
  -> Added 8 fourier coordinate features
Correcting trip duration mean...
  -> Corrected 0 null duration values
Creating occurrence flags...
  -> Departures occurred: 7,356,504/31,775,880 (23.2%)
  -> Arrivals occurred: 7,525,968/31,775,880 (23.7%)
Creating station embeddings mapping...
  -> Created mapping for 397 unique stations
Adding station embedding indices...
  -> Added embedding indices for 31,775,880 rows
✅ ENRICHMENT COMPLETE: 31,775,880 → 31,775,880 rows, 60 → 76 columns
   New columns: ['lat', 'lon', 'lat_z', 'lon_z', 'lat_s

In [ ]:

# Show sample of enriched data
print("Sample of enriched data:")
enriched_sample = df_feat.select([
    "station_id", "station_id_idx", "ts_start",
    "lat", "lon", "lat_z", "lon_z", 
    "lat_sin_1", "lat_cos_1", "lon_sin_1", "lon_cos_1",
    "trip_dur_mean_last_DT", "dur_is_null", 
    "has_dep", "has_arr", "dep_last_DT", "y_arrivals_next_DT"
]).head(10)

display(enriched_sample)

# Statistics on new flags
print(f"\n📈 Flag Statistics:")
print(f"  dur_is_null: {df_feat['dur_is_null'].sum():,} / {df_feat.height:,} ({df_feat['dur_is_null'].mean()*100:.1f}%)")
print(f"  has_dep: {df_feat['has_dep'].sum():,} / {df_feat.height:,} ({df_feat['has_dep'].mean()*100:.1f}%)")  
print(f"  has_arr: {df_feat['has_arr'].sum():,} / {df_feat.height:,} ({df_feat['has_arr'].mean()*100:.1f}%)")

# Coordinate statistics
print(f"\n🗺️  Coordinate Statistics:")
coord_stats = df_feat.select([
    pl.col("lat").min().alias("lat_min"),
    pl.col("lat").max().alias("lat_max"), 
    pl.col("lat").mean().alias("lat_mean"),
    pl.col("lon").min().alias("lon_min"),
    pl.col("lon").max().alias("lon_max"),
    pl.col("lon").mean().alias("lon_mean")
]).to_dicts()[0]

for key, val in coord_stats.items():
    print(f"  {key}: {val:.6f}")

# Station embedding info
print(f"\n🏢 Station Embedding Info:")
print(f"  Total stations: {len(station_id_to_idx)}")
print(f"  Embedding dimension: {embedding_dim}")
print(f"  Station ID range: {min(station_id_to_idx.keys())} - {max(station_id_to_idx.keys())}")
print(f"  Index range: 0 - {max(station_id_to_idx.values())}")

# Check trip duration correction  
print(f"\n⏱️ Trip Duration Correction Results:")
duration_stats = df_feat.group_by("dur_is_null").agg([
    pl.col("trip_dur_mean_last_DT").mean().alias("avg_duration"),
    pl.col("trip_dur_mean_last_DT").count().alias("count"),
    pl.col("dep_last_DT").mean().alias("avg_departures")
])
display(duration_stats)


QUEDA DROPPEAR ALGUNAS COLUMNAS INNECESARIAS DE WEATHER Y ESTAMOS!

In [9]:
df_feat.sample(5)

#count number of uniques ids 

df_feat.select(pl.col("station_id").unique().count())

station_id
u32
381


In [ ]:

# Save the enriched feature dataframe
enriched_path = '../../data/processed/trips_feat_eng_enriched.parquet'
df_feat.write_parquet(enriched_path)
print(f"✅ Saved enriched features to: {enriched_path}")
print(f"   Shape: {df_feat.shape}")
print(f"   Size: {os.path.getsize(enriched_path) / 1024 / 1024:.1f} MB")

# Create a summary of all enrichments applied
enrichment_summary = {
    'original_columns': len([col for col in df_feat.columns if col not in new_cols]),
    'new_columns': len(new_cols), 
    'total_columns': len(df_feat.columns),
    'total_rows': df_feat.height,
    'stations_with_embeddings': len(station_id_to_idx),
    'embedding_dimension': embedding_dim,
    'has_coordinates': 'lat' in df_feat.columns and 'lon' in df_feat.columns,
    'has_fourier_features': any('sin_' in col or 'cos_' in col for col in df_feat.columns),
    'duration_corrections': df_feat['dur_is_null'].sum(),
    'occurrence_flags_added': 'has_dep' in df_feat.columns and 'has_arr' in df_feat.columns
}

print(f"\n📋 ENRICHMENT SUMMARY:")
for key, value in enrichment_summary.items():
    print(f"   {key}: {value}")

print(f"\n🎯 DATA READY FOR MODELING!")
print("   ✅ Entity embeddings (station_id_idx)")
print("   ✅ Spatial features (lat/lon + fourier)")  
print("   ✅ Clean duration features")
print("   ✅ Binary occurrence flags")
print("   ✅ All original features preserved")


In [ ]:

print("The enriched dataset now includes:")
print("✅ station_id_idx: 0-based indices for embedding lookup")
print("✅ Mapping saved to: ../../data/processed/station_id_to_idx.pkl") 
print("✅ Embedding info saved to: ../../data/processed/embedding_info.pkl")

print(f"\n📊 Embedding Configuration:")
print(f"   Number of stations: {len(station_id_to_idx)}")
print(f"   Recommended embedding dimension: {embedding_dim}")
print(f"   Total embedding parameters: {len(station_id_to_idx) * embedding_dim:,}")

print(f"\n🏗️ Example PyTorch Usage:")
print("""
import torch
import torch.nn as nn
from embedding_utils import StationEmbedding, EcoBiciModel

# Create station embedding layer
station_emb = StationEmbedding(n_stations=381, embedding_dim=19)

# Example forward pass
station_indices = torch.tensor([0, 1, 2, 10, 25])  # batch of station indices
embeddings = station_emb(station_indices)
print(f"Embeddings shape: {embeddings.shape}")  # [5, 19]

# Full model with embeddings
model = EcoBiciModel(n_stations=381, n_features=50, embedding_dim=19)
""")

print("📁 See src/embedding_utils.py for complete implementation!")

# Show example of station indices
print(f"\n🔍 Sample Station ID → Index Mapping:")
sample_stations = list(station_id_to_idx.items())[:10]
for station_id, idx in sample_stations:
    print(f"   Station {station_id} → Index {idx}")

print("\n✨ Your data is now ready for advanced neural network modeling!")
print("   Including entity embeddings, spatial features, and clean time series data.")


In [3]:
df = pl.read_parquet('../../data/processed/df_feat_final_result.parquet')

In [6]:
print([col for col in df.columns if col.startswith('y_')])

['y_arrivals_next_DT']
